# Beluga AMCL Benchmark — APE Report

Report template for a completed Beluga benchmark run. Scoped to the outputs
of `beluga_benchmark.py` (`output.ape.zip`). Open from `examples/beluga/` and
execute all cells after a benchmark run.

In [ ]:
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from lambkin.data import access
from lambkin.data import evo as evo_data

# Change this path if your results live elsewhere
RESULTS_DIR = Path.cwd() / "results"

print(RESULTS_DIR)
APE_FILE = "output.ape.zip"

## Explore available data

Inspect what variants and iterations are available before plotting.

In [ ]:
iterations = access.iterations(RESULTS_DIR)

print(f"Total iterations: {len(iterations)}")
print()
for entry in iterations:
    label = ", ".join(f"{k}={v}" for k, v in sorted(vars(entry.params).items()))
    print(f"  {entry.variant} / iter {entry.iteration}  —  {label}")

## APE timeseries by variant

Each variant is drawn in a distinct color. Individual iterations are shown
at reduced opacity; the per-variant mean is overlaid in bold.

In [ ]:
series = evo_data.series(RESULTS_DIR, APE_FILE)

by_variant = defaultdict(list)
for entry in series:
    by_variant[entry.variant].append(entry)

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for (_variant, entries), color in zip(sorted(by_variant.items()), colors, strict=False):
    label = ", ".join(f"{k}={v}" for k, v in sorted(vars(entries[0].params).items()))
    for entry in entries:
        ax.plot(entry.time, entry.error, color=color, alpha=0.3, linewidth=0.8)
    t_min = max(e.time[0] for e in entries)
    t_max = min(e.time[-1] for e in entries)
    t_grid = np.linspace(t_min, t_max, 300)
    mean_error = np.mean([np.interp(t_grid, e.time, e.error) for e in entries], axis=0)
    ax.plot(t_grid, mean_error, color=color, linewidth=2, label=label)

ax.set_xlabel("Time (s)")
ax.set_ylabel("APE (m)")
ax.set_title("Absolute Pose Error — timeseries by variant")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.savefig(RESULTS_DIR / "report_ape_series.png", dpi=150)
plt.show()

## Stats summary table

RMSE, mean, and max APE aggregated across iterations for each variant.

In [ ]:
all_stats = evo_data.stats(RESULTS_DIR, APE_FILE)

by_variant_stats = defaultdict(list)
for entry in all_stats:
    label = ", ".join(f"{k}={v}" for k, v in sorted(vars(entry.params).items()))
    by_variant_stats[label].append(entry)

header = (
    f"{'Variant':<40} {'N':>4}"
    f" {'RMSE mean':>10} {'RMSE std':>10} {'Mean':>10} {'Max':>10}"
)
print(header)
print("-" * len(header))
for label in sorted(by_variant_stats):
    entries = by_variant_stats[label]
    rmse = [e.rmse for e in entries]
    print(
        f"{label:<40} {len(entries):>4}"
        f" {np.mean(rmse):>10.4f} {np.std(rmse):>10.4f}"
        f" {np.mean([e.mean for e in entries]):>10.4f}"
        f" {np.mean([e.max for e in entries]):>10.4f}"
    )

## RMSE comparison across variants

Bar chart comparing RMSE per variant, with error bars showing ± std across iterations.

In [ ]:
labels = sorted(by_variant_stats.keys())
rmse_means = [np.mean([e.rmse for e in by_variant_stats[lbl]]) for lbl in labels]
rmse_stds = [np.std([e.rmse for e in by_variant_stats[lbl]]) for lbl in labels]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.2), 4))
x = np.arange(len(labels))
ax.bar(x, rmse_means, yerr=rmse_stds, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("RMSE (m)")
ax.set_title("APE RMSE by variant (mean ± std across iterations)")
fig.tight_layout()
plt.savefig(RESULTS_DIR / "report_rmse_bars.png", dpi=150)
plt.show()